## Pipeline ENADE - Mineração de Dados Socioeconômicos
**TCC · Giovanna Naves Ribeiro · PUC Minas · 2025**
*Orientadora: Maria Inês Martins*

> Este notebook foi desenvolvido com auxílio do **Claude (Anthropic)**.

---

> **Download dos microdados:** https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enade
> Descompacte o ZIP e organize os arquivos em `data/{ano}/microdados{ano}_arq*.csv` (uma subpasta por ano), na MESMA pasta onde está este notebook.

### Sobre a unidade de análise deste pipeline
O ENADE não fornece um identificador de aluno em comum entre os diferentes arquivos `microdados{ano}_arqN.csv` de um mesmo ano. Por isso, cada arquivo é primeiro resumido por curso (médias e proporções) e só depois essas agregações são unidas por `CO_CURSO`. Como resultado, cada linha do dado final representa **um curso em um ano**, não um aluno, o que ainda permite analisar perfil demográfico por curso, nota média por curso e a relação entre renda e nota comparando cursos entre si (inclusive por área).

## Etapa 1 - Instalação de dependências

In [ ]:
!pip install pyarrow scipy seaborn tqdm --quiet
print("Dependências instaladas.")

Dependências instaladas.


## Etapa 2 - Diretórios do projeto
Os caminhos abaixo são sempre relativos à pasta onde este notebook está salvo - não dependem de usuário ou máquina. Basta que a pasta `data` esteja ao lado deste arquivo `.ipynb`.

In [2]:
import os, json, logging, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("enade")


def _detectar_base_dir():
    candidatos = [Path.cwd()]
    val = globals().get("__vsc_ipynb_file__") 
    if val:
        candidatos.append(Path(val).resolve().parent)
    for cand in candidatos:
        if (cand / "data").exists():
            return cand.resolve()
    return candidatos[0].resolve()

BASE_DIR    = _detectar_base_dir()
DATA_DIR    = BASE_DIR / "data"
OUTPUTS_DIR = BASE_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Pasta 'data' não encontrada em {BASE_DIR}.\n"
        "Coloque este notebook na mesma pasta que contém a pasta 'data' "
        "(com uma subpasta por ano dentro dela, ex.: data/2019, data/2021, ...)."
    )

print(f"Diretório base:   {BASE_DIR}")
print(f"Dados (entrada):  {DATA_DIR}")
print(f"Saídas (outputs): {OUTPUTS_DIR}")


Diretório base:   D:\TCC_ENADE
Dados (entrada):  D:\TCC_ENADE\data
Saídas (outputs): D:\TCC_ENADE\outputs


## Etapa 3 - Mapeamentos de variáveis, labels e parâmetros do questionário

In [3]:
ARQUIVO_VARIAVEIS = {
    "arq1":  ["CO_IES","CO_CATEGAD","CO_ORGACAD","CO_GRUPO","CO_MODALIDADE",
              "CO_UF_CURSO","CO_REGIAO_CURSO"], # CO_CATEGAD = codigo da categoria administrativa, CO_MODALIDADE = EaD ou Presencial. Municipio, UF e Região
    "arq3":  ["TP_PRES","NT_GER","NT_FG","NT_OBJ_FG","NT_DIS_FG",
              "NT_CE","NT_OBJ_CE","NT_DIS_CE"],
    "arq5":  ["TP_SEXO"],                  # sexo
    "arq6":  ["NU_IDADE"],                 # idade
    "arq8":  ["QE_I02"],                   # raça/cor
    "arq10": ["QE_I04"],                   # escolaridade do pai
    "arq11": ["QE_I05"],                   # escolaridade da mãe
    "arq14": ["QE_I08"],                   # renda familiar
    "arq16": ["QE_I10"],                   # situação de trabalho
    "arq17": ["QE_I11"],                   # financiamento do curso
    "arq21": ["QE_I15"],                   # cotas/reserva de vagas
    "arq23": ["QE_I17"],                   # tipo de escola no ensino médio
    "arq27": ["QE_I21"],                   # intercâmbio
    "arq28": ["QE_I22"],                   # livros lidos no ano
    "arq29": ["QE_I23"],                   # horas de estudo/semana
}

VARS_DESEMPENHO = ["NT_GER","NT_FG","NT_OBJ_FG","NT_DIS_FG","NT_CE","NT_OBJ_CE","NT_DIS_CE"]
# Para cada variável de desempenho, calculamos por curso tanto a MÉDIA quanto
# a MODA. Cada uma gera duas colunas no resultado final: NT_GER_MEDIA e NT_GER_MODA, etc.
VARS_DESEMPENHO_MEDIA = [f"{v}_MEDIA" for v in VARS_DESEMPENHO]
VARS_DESEMPENHO_MODA  = [f"{v}_MODA" for v in VARS_DESEMPENHO]

# Como cada variável é resumida por curso na etapa de agregação:

# 1) VARS_NUMERICAS_MEDIA → só MÉDIA 
# 2) VARS_NUMERICAS_MEDIA_MODA → MÉDIA + MODA
# 3) VARS_CATEGORICAS_ABSOLUTA → só CONTAGEM ABSOLUTA por categoria, mantém os números absolutos na visualização.
# 4) VARS_BINARIAS_PERCENTUAL → só o PERCENTUAL de uma categoria "positiva" (o restante é implicitamente o complemento): modalidade (% presencial,
#    resto é EAD), sexo (% feminino, resto é masculino), intercâmbio (% sim, resto é não).
# 5) GRUPOS_CONTAGEM → variáveis com muitas categorias originais que agregamos em grupos maiores e mais interpretáveis, mantendo a
#    CONTAGEM ABSOLUTA de cada grupo (categoria administrativa, financiamento).

VARS_NUMERICAS_MEDIA = [
    "NU_IDADE",
    "QE_I08_NUM",
    "QE_I04_NUM",
    "QE_I05_NUM",
    "QE_I10_NUM",
    "QE_I17_NUM",
]

VARS_NUMERICAS_MEDIA_MODA = [
    "QE_I22_NUM",
    "QE_I23_NUM",
]

VARS_CATEGORICAS_ABSOLUTA = [
    "QE_I02",
    "QE_I15",
]

VARS_BINARIAS_PERCENTUAL = {
    "CO_MODALIDADE": {"positivo": "1", "coluna": "PCT_PRESENCIAL"},
    "TP_SEXO":       {"positivo": "F", "coluna": "PCT_SEXO_FEMININO"},
    "QE_I21":        {"positivo": "A", "coluna": "PCT_INTERCAMBIO_SIM"},
}

GRUPOS_CONTAGEM = {
    "CO_CATEGAD": {
        "1": "PUBLICA", "2": "PUBLICA", "3": "PUBLICA",
        "4": "PRIVADA", "5": "PRIVADA",
        "7": "OUTROS",  "8": "OUTROS",
    },
    "QE_I11": {
        "A": "NENHUM", "B": "NENHUM",
        "C": "PROUNI", "D": "PROUNI",
        "E": "FIES",
        "F": "PROUNI_FIES",
        "G": "BOLSA_GOV", "H": "BOLSA_GOV",
        "I": "OUTROS", "J": "OUTROS", "K": "OUTROS",
    },
}
GRUPOS_CONTAGEM_PREFIXO_COLUNA = {
    "CO_CATEGAD": "QTD_CATEGAD_",
    "QE_I11":     "QTD_FINANC_",
}

VARS_IDENTIDADE_CURSO = [
    "CO_IES", "CO_ORGACAD", "CO_GRUPO", "CO_UF_CURSO", "CO_REGIAO_CURSO",
]

QE_AUSENTES     = ["8","9",8,9,""]
TP_PRES_REGULAR = 555

LABELS = {
    "TP_SEXO":    {"M":"Masculino","F":"Feminino"},
    "QE_I02":     {"A":"Branca","B":"Preta","C":"Amarela","D":"Parda","E":"Indígena","F":"Não declarado"},
    "QE_I08":     {"A":"Até 1,5 SM","B":"1,5–3 SM","C":"3–4,5 SM","D":"4,5–6 SM","E":"6–10 SM","F":"10–30 SM","G":">30 SM"},
    "QE_I08_ORD": {"A":1,"B":2,"C":3,"D":4,"E":5,"F":6,"G":7},
    "QE_I04":     {"A":"Nenhuma","B":"Fund.1–5","C":"Fund.6–9","D":"Médio","E":"Superior","F":"Pós-grad."},
    "QE_I04_ORD": {"A":0,"B":1,"C":2,"D":3,"E":4,"F":5},
    "QE_I05":     {"A":"Nenhuma","B":"Fund.1–5","C":"Fund.6–9","D":"Médio","E":"Superior","F":"Pós-grad."},
    "QE_I05_ORD": {"A":0,"B":1,"C":2,"D":3,"E":4,"F":5},
    "QE_I10":     {"A":"Não trabalha","B":"Eventualmente","C":"Até 20h","D":"21–39h","E":"40h+"},
    "QE_I10_ORD": {"A":0,"B":10,"C":20,"D":30,"E":40},
    "QE_I17":     {"A":"Todo público","B":"Todo privado","C":"Exterior","D":"Maior p. público","E":"Maior p. privado","F":"BR+exterior"},
    # Ordem: Todo público < Maior p. público < Maior p. privado < Todo privado < BR+exterior < Exterior (do mais "público" ao mais
    # "fora do sistema público brasileiro").
    "QE_I17_ORD": {"A":0,"D":1,"E":2,"B":3,"F":4,"C":5},
    "QE_I15":     {"A":"Não","B":"Étnico-racial","C":"Renda","D":"Escola pública","E":"Combinação","F":"Outro"},
    "QE_I11":     {"A":"Gratuito","B":"Nenhum (pago)","C":"ProUni integral","D":"ProUni parcial",
                   "E":"FIES","F":"ProUni+FIES","G":"Bolsa gov.","H":"Bolsa IES","I":"Bolsa outra",
                   "J":"Financ. IES","K":"Financ. bancário"},
    "QE_I12":     {"A":"Nenhum","B":"Moradia","C":"Alimentação","D":"Moradia+alim.","E":"Permanência","F":"Outro"},
    "QE_I21":     {"A":"Sim","B":"Não"},
    "QE_I22":     {"A":"Nenhum","B":"1–2","C":"3–5","D":"6–8","E":"Mais de 8"},
    "QE_I22_ORD": {"A":0,"B":1,"C":2,"D":3,"E":4},
    "QE_I23":     {"A":"Nenhuma","B":"1–3h","C":"4–7h","D":"8–12h","E":"Mais de 12h"},
    "QE_I23_ORD": {"A":0,"B":1,"C":2,"D":3,"E":4},
    "CO_CATEGAD":  {1:"Pública Federal",2:"Pública Estadual",3:"Pública Municipal",
                    4:"Privada c/ fins luc.",5:"Privada s/ fins luc.",7:"Especial",8:"Comunitária"},
    "CO_CATEGAD_GRUPO": {"PUBLICA":"Pública (Federal/Estadual/Municipal)",
                          "PRIVADA":"Privada (com/sem fins lucrativos)",
                          "OUTROS":"Especial/Comunitária"},
    "CO_MODALIDADE":{0:"EAD",1:"Presencial"},
    "QE_I11_GRUPO": {"NENHUM":"Nenhum","PROUNI":"ProUni","FIES":"FIES",
                      "PROUNI_FIES":"ProUni+FIES","BOLSA_GOV":"Bolsa governamental","OUTROS":"Outros"},
}

CO_GRUPO_NOMES = {
    1:"Administração",2:"Direito",5:"Medicina Veterinária",6:"Odontologia",12:"Medicina",
    13:"Ciências Econômicas",17:"Agronomia",18:"Psicologia",19:"Farmácia",
    21:"Arquitetura e Urbanismo",22:"Ciências Contábeis",23:"Enfermagem",26:"Design",
    27:"Fonoaudiologia",28:"Nutrição",29:"Turismo",35:"Educação Física",36:"Fisioterapia", 
    38:"Serviço Social",
    51:"Zootecnia",52:"Terapia Ocupacional",55:"Biomedicina",67:"Secretariado Executivo",
    69:"Tecnologia em Radiologia",71:"Tecnologia em Alimentos",72:"Tecnologia em Análise e Desenvolvimento de Sistemas",
    73:"Tecnologia em Automação Industrial",74:"Tecnologia em Construção de Edifícios",75:"Tecnologia em Fabricação Mecânica",
    76:"Tecnologia em Gestão da Produção Industrial",77:"Tecnologia em Manutenção Industrial",78:"Tecnologia em Processos Químicos",
    79:"Tecnologia em Redes de Computadores",80:"Tecnologia em Saneamento Ambiental",81:"Relações Internacionais",
    83:"Tecnologia em Design de Moda",84:"Tecnologia em Marketing",85:"Tecnologia em Processos Gerenciais",
    86:"Tecnologia em Gestão de Recursos Humanos",87:"Tecnologia em Gestão Financeira",
    88:"Tecnologia em Gastronomia",90:"Tecnologia em Agronegócios",
    91:"Tecnologia em Gestão Hospitalar",92:"Tecnologia em Gestão Ambiental",93:"Tecnologia em Gestão Comercial",
    94:"Tecnologia em Logística",95:"Tecnologia em Estética e Cosmética",
    100:"Administração Pública",101:"Teologia",102:"Tecnologia em Comércio Exterior",
    103:"Tecnologia em Design de Interiores",104:"Tecnologia em Design Gráfico",
    105:"Tecnologia em Gestão da Qualidade",106:"Tecnologia em Gestão Pública",
    701:"Matemática (Bacharelado)",
    702:"Matemática (Licenciatura)",803:"Jornalismo",804:"Publicidade e Propaganda",901:"Letras (Bacharelado)",
    902:"Letras (Licenciatura)",903:"Letras-Português (Bacharelado)",904:"Letras-Português (Licenciatura)",
    905:"Letras-Português e Inglês (Licenciatura)",906:"Letras-Português e Espanhol (Licenciatura)",
    1401:"Física (Bacharelado)",1402:"Física (Licenciatura)",1501:"Química (Bacharelado)",
    1502:"Química (Licenciatura)",1503:"Química (Atribuições Tecnológicas)",1601:"Biologia (Bacharelado)",1602:"Biologia (Licenciatura)",2001:"Pedagogia (Licenciatura)",
    2401:"História (Bacharelado)",2402:"História (Licenciatura)",2501:"Artes Visuais (Licenciatura)",3001:"Geografia (Bacharelado)",
    3002:"Geografia (Licenciatura)",3201:"Filosofia (Bacharelado)",3202:"Filosofia (Licenciatura)",
    3501:"Educação Física (Licenciatura)",3502:"Educação Física (Licenciatura)",4003:"Engenharia da Computação",
    4004:"Computação (Bacharelado)",4005:"Computação (Licenciatura)",
    4006:"Sistemas de Informação",4007:"Engenharia da Computação",4301:"Música (Licenciatura)",
    5401:"Ciências Sociais (Bacharelado)",5402:"Ciências Sociais (Licenciatura)",5707:"Engenharia Geológica",
    5708:"Engenharia de Agrimensura",5709:"Engenharia Cartográfica",5710:"Engenharia Civil",5711:"Engenharia de Recursos",
    5712:"Engenharia Sanitária",5806:"Engenharia Elétrica",5807:"Engenharia Industrial",5808:"Engenharia Eletrotécnica",
    5809:"Engenharia de Computação",5811:"Engenharia de Redes de Computadores",5812:"Engenharia Eletrônica",
    5813:"Engenharia Mecatrônica",5814:"Engenharia de Controle e Automação",5815:"Engenharia",
    5901:"Engenharia Industrial",5902:"Engenharia Mecânica",5903:"Engenharia Aeroespacial",5904:"Engenharia Aeronáutica",
    5905:"Engenharia Automotiva",5906:"Engenharia Naval",6002:"Engenharia de Alimentos",6005:"Engenharia Bioquímica",
    6006:"Engenharia",6007:"Engenharia Industrial",6008:"Engenharia Química",6009:"Engenharia de Alimentos",6011:"Engenharia Têxtil",
    6106:"Engenharia de Materiais",6107:"Engenharia Física",6108:"Engenharia Metalúrgica",6109:"Engenharia de Materiais",
    6110:"Engenharia de Materiais",
    6208:"Engenharia de Produção",6209:"Engenharia de Produção Mecânica",6210:"Engenharia de Produção",
    6211:"Engenharia de Produção",6213:"Engenharia de Produção de Materiais",6214:"Engenharia de Produção",
    6306:"Engenharia",6307:"Engenharia Ambiental",6308:"Engenharia Industrial",6309:"Engenharia de Minas",
    6310:"Engenharia de Petróleo",6404:"Engenharia Agrícola",6405:"Engenharia Florestal",6406:"Engenharia de Pesca",
    6407:"Letras - Inglês",6409:"Tecnologia em Gestão da Tecnologia da Informação",
    6410:"Tecnologia em Segurança do Trabalho",6411:"Engenharia de Computação I",
}

NOMES_QTD = {
    "QE_I02": {
        "A": "QTD_RACA_BRANCA", "B": "QTD_RACA_PRETA", "C": "QTD_RACA_AMARELA",
        "D": "QTD_RACA_PARDA", "E": "QTD_RACA_INDIGENA", "F": "QTD_RACA_NAO_DECLARADO",
    },
    "QE_I15": {
        "A": "QTD_COTAS_NAO", "B": "QTD_COTAS_ETNICO_RACIAL", "C": "QTD_COTAS_RENDA",
        "D": "QTD_COTAS_ESCOLA_PUBLICA", "E": "QTD_COTAS_COMBINACAO_DE_CRITERIOS", "F": "QTD_COTAS_OUTRO_SISTEMA",
    },
}

print("Configuração carregada.")
print(f"   Arquivos de microdados usados: {list(ARQUIVO_VARIAVEIS.keys())}")


Configuração carregada.
   Arquivos de microdados usados: ['arq1', 'arq3', 'arq5', 'arq6', 'arq8', 'arq10', 'arq11', 'arq14', 'arq16', 'arq17', 'arq21', 'arq23', 'arq27', 'arq28', 'arq29']


## Etapa 4 - Descoberta automática dos anos disponíveis
Todas as subpastas de `data/` nomeadas apenas com o ano (ex.: `data/2019`) são detectadas automaticamente. Não há mais um intervalo fixo `ANO_INICIO`/`ANO_FIM` para editar: para incluir um novo ano, basta adicionar a pasta correspondente.

In [4]:
ANOS_DISPONIVEIS = sorted([
    int(p.name) for p in DATA_DIR.iterdir()
    if p.is_dir() and p.name.isdigit()
])

if not ANOS_DISPONIVEIS:
    raise FileNotFoundError(
        f"Nenhuma subpasta de ano encontrada em {DATA_DIR}.\n"
        "Crie uma subpasta por ano (ex.: data/2019) com os arquivos "
        "microdados{ano}_arq*.csv dentro."
    )

print(f"Anos encontrados ({len(ANOS_DISPONIVEIS)}): {ANOS_DISPONIVEIS}\n")

for ano in ANOS_DISPONIVEIS:
    pasta = DATA_DIR / str(ano)
    arquivos = [p for p in pasta.glob("*.*") if p.suffix.lower() in (".csv", ".txt")]
    print(f"   {ano}: {len(arquivos)} arquivos")


Anos encontrados (13): [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023]

   2010: 27 arquivos
   2011: 27 arquivos
   2012: 27 arquivos
   2013: 31 arquivos
   2014: 42 arquivos
   2015: 32 arquivos
   2016: 32 arquivos
   2017: 42 arquivos
   2018: 32 arquivos
   2019: 32 arquivos
   2021: 43 arquivos
   2022: 33 arquivos
   2023: 32 arquivos


## Etapa 5 - Funções de ingestão e agregação por curso
Esta célula apenas **define** as funções; nenhum dado é processado ainda.

In [5]:
NA_VALUES = ["", ".", "..", " ", "NA", "N/A"]

def _localizar_arquivo(sources, ano, nome_arq):
    candidatos = [
        f"microdados{ano}_{nome_arq}.csv", f"microdados{ano}_{nome_arq}.txt",
        f"microdados_{ano}_{nome_arq}.csv", f"microdados_{ano}_{nome_arq}.txt",
        f"{nome_arq}.csv", f"{nome_arq}.txt",
        f"MICRODADOS{ano}_{nome_arq.upper()}.CSV", f"MICRODADOS{ano}_{nome_arq.upper()}.TXT",
    ]
    return next((sources[c] for c in candidatos if c in sources), None)


def _ler_csv(csv_path, variaveis):
    cols_ok = ["NU_ANO", "CO_CURSO"] + variaveis
    for enc in ("latin-1", "utf-8", "utf-8-sig"):
        for quoting in (0, 3):  # 0=QUOTE_MINIMAL, 3=QUOTE_NONE
            try:
                df = pd.read_csv(
                    csv_path, sep=";", encoding=enc,
                    usecols=lambda c: c.strip() in cols_ok,
                    low_memory=False, quoting=quoting,
                    na_values=NA_VALUES, keep_default_na=False,
                )
                df.columns = df.columns.str.strip()
                for col in df.select_dtypes(include="object").columns:
                    df[col] = df[col].str.strip().str.strip('"')
                return df
            except Exception:
                continue
    return pd.DataFrame()


def _codificar_ordinais(df):
    df = df.copy()
    mapas = {"QE_I08": LABELS["QE_I08_ORD"], "QE_I04": LABELS["QE_I04_ORD"],
             "QE_I05": LABELS["QE_I05_ORD"], "QE_I10": LABELS["QE_I10_ORD"],
             "QE_I17": LABELS["QE_I17_ORD"],
             "QE_I22": LABELS["QE_I22_ORD"], "QE_I23": LABELS["QE_I23_ORD"]}
    for col, mapa in mapas.items():
        if col in df.columns:
            df[f"{col}_NUM"] = df[col].astype(str).str.strip().str.upper().map(mapa)
    return df


def _tratar_ausentes_qe(df):
    df = df.copy()
    qe_cols = [c for c in df.columns if c.startswith("QE_I")]
    ausentes_str = [str(v) for v in QE_AUSENTES]
    for col in qe_cols:
        mask = df[col].astype(str).str.strip().isin(ausentes_str)
        df.loc[mask, col] = np.nan
    return df

def _normalizar_categorica(serie):
    s = serie.astype(str).str.strip().str.upper()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace({"NAN": np.nan, "": np.nan})
    return s

def _moda(serie):
    m = serie.mode()
    return m.iloc[0] if not m.empty else np.nan


def agregar_arquivo_por_curso(df_arq, nome_arq):
    if df_arq.empty or "CO_CURSO" not in df_arq.columns:
        return pd.DataFrame()

    df = df_arq.copy()

    if nome_arq == "arq3":
        if "TP_PRES" in df.columns:
            df = df[df["TP_PRES"] == TP_PRES_REGULAR]
        for col in VARS_DESEMPENHO:
            if col in df.columns:
                df[col] = (df[col].astype(str).str.replace(",", ".", regex=False)
                           .pipe(pd.to_numeric, errors="coerce"))

    df = _tratar_ausentes_qe(df)
    df = _codificar_ordinais(df)

    cols_categoricas = (set(VARS_CATEGORICAS_ABSOLUTA)
                         | set(GRUPOS_CONTAGEM)
                         | set(VARS_BINARIAS_PERCENTUAL))
    for col in cols_categoricas:
        if col in df.columns:
            df[col] = _normalizar_categorica(df[col])

    grp = df.groupby(["NU_ANO", "CO_CURSO"])
    partes = [grp.size().rename(f"N_{nome_arq}")]

    for col in df.columns:
        if col in ("NU_ANO", "CO_CURSO"):
            continue

        if col in VARS_DESEMPENHO:
            partes.append(grp[col].mean().rename(f"{col}_MEDIA"))
            partes.append(grp[col].agg(_moda).rename(f"{col}_MODA"))

        elif col in VARS_NUMERICAS_MEDIA_MODA:
            partes.append(grp[col].mean().rename(f"{col}_MEDIA"))
            partes.append(grp[col].agg(_moda).rename(f"{col}_MODA"))

        elif col in VARS_NUMERICAS_MEDIA:
            partes.append(grp[col].mean().rename(col))

        elif col in GRUPOS_CONTAGEM:
            # Reagrupa categorias originais em grupos maiores e conta o absoluto de alunos por grupo, dentro de cada curso/ano.
            mapa = GRUPOS_CONTAGEM[col]
            serie_grupo = df[col].map(mapa)
            contagem = (pd.concat([df[["NU_ANO", "CO_CURSO"]], serie_grupo.rename(col)], axis=1)
                          .groupby(["NU_ANO", "CO_CURSO"])[col]
                          .value_counts().unstack(fill_value=0))
            prefixo = GRUPOS_CONTAGEM_PREFIXO_COLUNA.get(col, f"QTD_{col}_")
            contagem.columns = [f"{prefixo}{c}" for c in contagem.columns]
            partes.append(contagem)

        elif col in VARS_BINARIAS_PERCENTUAL:
            cfg = VARS_BINARIAS_PERCENTUAL[col]
            flag = pd.Series(np.where(df[col].isna(), np.nan,
                                       (df[col] == cfg["positivo"]).astype(float)),
                              index=df.index, name="flag")
            tmp = pd.concat([df[["NU_ANO", "CO_CURSO"]], flag], axis=1)
            pct = tmp.groupby(["NU_ANO", "CO_CURSO"])["flag"].mean().rename(cfg["coluna"])
            partes.append(pct)

        elif col in VARS_CATEGORICAS_ABSOLUTA:
            contagem = (df.groupby(["NU_ANO", "CO_CURSO"])[col]
                          .value_counts().unstack(fill_value=0))
            nomes_amigaveis = NOMES_QTD.get(col, {})
            contagem.columns = [nomes_amigaveis.get(c, f"QTD_{col}_{c}") for c in contagem.columns]
            partes.append(contagem)

        elif col in VARS_IDENTIDADE_CURSO:
            partes.append(grp[col].agg(_moda).rename(col))

    return pd.concat(partes, axis=1).reset_index()


def ingerir_e_agregar(ano):
    pasta = DATA_DIR / str(ano)
    sources = {p.name: p for p in pasta.glob("*.*") if p.suffix.lower() in (".csv", ".txt")}
    if not sources:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em {pasta}")

    log.info(f"[Ingestor] ENADE {ano} — {len(sources)} arquivos disponíveis")

    df_curso, lidos, ausentes = None, 0, []
    for nome_arq, variaveis in ARQUIVO_VARIAVEIS.items():
        caminho = _localizar_arquivo(sources, ano, nome_arq)
        if caminho is None:
            ausentes.append(nome_arq)
            continue
        df_arq = _ler_csv(caminho, variaveis)
        if df_arq.empty:
            continue
        df_arq["NU_ANO"] = ano

        df_agg = agregar_arquivo_por_curso(df_arq, nome_arq)
        if df_agg.empty:
            continue
        lidos += 1
        df_curso = df_agg if df_curso is None else df_curso.merge(
            df_agg, on=["NU_ANO", "CO_CURSO"], how="outer")

    if df_curso is None or df_curso.empty:
        raise ValueError(f"Nenhum arquivo agregado para {ano}.")
    if ausentes:
        log.warning(f"[Ingestor] Arquivos não encontrados (ignorados): {', '.join(ausentes)}")

    if "N_arq3" in df_curso.columns:
        df_curso = df_curso.rename(columns={"N_arq3": "N_ALUNOS"})

    cols_qtd = [c for c in df_curso.columns if c.startswith("QTD_")]
    if cols_qtd:
        df_curso[cols_qtd] = df_curso[cols_qtd].fillna(0).astype(int)

    log.info(f"[Ingestor] {lidos} arquivos agregados → {len(df_curso):,} cursos × {df_curso.shape[1]} colunas")
    return df_curso


print("Funções de ingestão/agregação por curso definidas.")


Funções de ingestão/agregação por curso definidas.


## Etapa 6 - Teste de ingestão para um ano

In [6]:
ANO_TESTE = ANOS_DISPONIVEIS[-1]

print(f"Testando ingestão/agregação para {ANO_TESTE}...")
df_curso_bruto = ingerir_e_agregar(ANO_TESTE)

print(f"\nShape (cursos × colunas): {df_curso_bruto.shape}")
print(f"Colunas (amostra): {list(df_curso_bruto.columns[:30])}")
df_curso_bruto.head(5)


Testando ingestão/agregação para 2023...


[Ingestor] ENADE 2023 — 32 arquivos disponíveis
[Ingestor] 15 arquivos agregados → 9,812 cursos × 70 colunas



Shape (cursos × colunas): (9812, 70)
Colunas (amostra): ['NU_ANO', 'CO_CURSO', 'N_arq1', 'CO_IES', 'QTD_CATEGAD_PUBLICA', 'QTD_CATEGAD_OUTROS', 'QTD_CATEGAD_PRIVADA', 'CO_ORGACAD', 'CO_GRUPO', 'PCT_PRESENCIAL', 'CO_UF_CURSO', 'CO_REGIAO_CURSO', 'N_ALUNOS', 'NT_GER_MEDIA', 'NT_GER_MODA', 'NT_FG_MEDIA', 'NT_FG_MODA', 'NT_OBJ_FG_MEDIA', 'NT_OBJ_FG_MODA', 'NT_DIS_FG_MEDIA', 'NT_DIS_FG_MODA', 'NT_CE_MEDIA', 'NT_CE_MODA', 'NT_OBJ_CE_MEDIA', 'NT_OBJ_CE_MODA', 'NT_DIS_CE_MEDIA', 'NT_DIS_CE_MODA', 'N_arq5', 'PCT_SEXO_FEMININO', 'N_arq6']


,NU_ANO,CO_CURSO,N_arq1,CO_IES,QTD_CATEGAD_PUBLICA,QTD_CATEGAD_OUTROS,QTD_CATEGAD_PRIVADA,CO_ORGACAD,CO_GRUPO,PCT_PRESENCIAL,...,N_arq23,QE_I17_NUM,N_arq27,PCT_INTERCAMBIO_SIM,N_arq28,QE_I22_NUM_MEDIA,QE_I22_NUM_MODA,N_arq29,QE_I23_NUM_MEDIA,QE_I23_NUM_MODA
0,2023,3,33,1,33,0,0,10028,5710,1.0,...,33,1.516129,33,0.870968,33,1.451613,1.0,33,2.516129,4.0
1,2023,9,42,1,42,0,0,10028,17,1.0,...,42,1.166667,42,0.857143,42,1.166667,1.0,42,2.095238,1.0
2,2023,10,13,1,13,0,0,10028,6405,1.0,...,13,0.500000,13,0.750000,13,1.416667,1.0,13,1.666667,1.0
3,2023,12,78,1,78,0,0,10028,12,1.0,...,78,1.576923,78,0.935897,78,1.358974,1.0,78,2.910256,4.0
4,2023,16,28,1,28,0,0,10028,5806,1.0,...,28,1.074074,28,0.851852,28,1.629630,1.0,28,2.481481,2.0


## Etapa 7 - Engenharia de atributos por curso

In [7]:
def adicionar_derivadas_curso(df):
    df = df.copy()

    if "CO_GRUPO" in df.columns:
        df["NOME_CURSO"] = df["CO_GRUPO"].map(CO_GRUPO_NOMES).fillna(
            df["CO_GRUPO"].astype(str).apply(lambda x: f"Grupo {x}"))

    if "QE_I08_NUM" in df.columns:
        conds = [df["QE_I08_NUM"] <= 2, df["QE_I08_NUM"].between(3, 4), df["QE_I08_NUM"] >= 5]
        df["FAIXA_RENDA_SIMPL"] = np.select(
            conds, ["Baixa (≤3 SM)", "Média (3–6 SM)", "Alta (>6 SM)"], default="Não informado")

    comps = []
    for col, div in [("QE_I04_NUM", 5), ("QE_I05_NUM", 5), ("QE_I23_NUM_MEDIA", 4), ("QE_I22_NUM_MEDIA", 4)]:
        if col in df.columns:
            comps.append(df[col] / div)
    if comps:
        df["IDX_CAPITAL_CULT"] = pd.concat(comps, axis=1).mean(axis=1)

    return df


df_curso_eng = adicionar_derivadas_curso(df_curso_bruto)

derivadas = ["NOME_CURSO", "FAIXA_RENDA_SIMPL", "IDX_CAPITAL_CULT",
             "PCT_SEXO_FEMININO", "PCT_PRESENCIAL", "PCT_INTERCAMBIO_SIM"]
print("Variáveis derivadas/percentuais presentes:", [c for c in derivadas if c in df_curso_eng.columns])
print(f"Shape final: {df_curso_eng.shape}  (cada linha = 1 curso em {ANO_TESTE})")
df_curso_eng[[c for c in derivadas if c in df_curso_eng.columns] + ["NT_GER_MEDIA", "NT_GER_MODA"]].dropna(subset=["NT_GER_MEDIA"]).head(5)


Variáveis derivadas/percentuais presentes: ['NOME_CURSO', 'FAIXA_RENDA_SIMPL', 'IDX_CAPITAL_CULT', 'PCT_SEXO_FEMININO', 'PCT_PRESENCIAL', 'PCT_INTERCAMBIO_SIM']
Shape final: (9812, 73)  (cada linha = 1 curso em 2023)


,NOME_CURSO,FAIXA_RENDA_SIMPL,IDX_CAPITAL_CULT,PCT_SEXO_FEMININO,PCT_PRESENCIAL,PCT_INTERCAMBIO_SIM,NT_GER_MEDIA,NT_GER_MODA
0,Engenharia Civil,Média (3–6 SM),0.606048,0.212121,1.0,0.870968,59.612903,31.0
1,Agronomia,Média (3–6 SM),0.512202,0.309524,1.0,0.857143,59.377778,60.6
2,Engenharia Florestal,Não informado,0.509375,0.692308,1.0,0.750000,45.709091,43.6
3,Medicina,Não informado,0.632853,0.525641,1.0,0.935897,69.443590,61.5
4,Engenharia Elétrica,Média (3–6 SM),0.579167,0.214286,1.0,0.851852,51.273913,45.3


## Etapa 8 - Correlação socioeconômica × desempenho, entre cursos

In [10]:
def correlacao_geral_curso(df):
    vars_soc = {
        "QE_I08_NUM": "Renda", 
        "QE_I04_NUM": "Escol.Pai", 
        "QE_I05_NUM": "Escol.Mãe",
        "QE_I10_NUM": "Trabalho", 
        "QE_I17_NUM": "Escola.EM",
        "QE_I22_NUM_MEDIA": "Leitura", 
        "QE_I23_NUM_MEDIA": "H.Estudo",
        "IDX_CAPITAL_CULT": "Cap.Cultural",
        "NU_IDADE": "Idade",
        "PCT_SEXO_FEMININO": "% Feminino",
        "PCT_INTERCAMBIO_SIM": "% Intercâmbio"
    }
    vars_nota = [v for v in VARS_DESEMPENHO_MEDIA if v in df.columns]
    rows = []
    for col_s, label in vars_soc.items():
        if col_s not in df.columns:
            continue
        row = {"Variável": label}
        for vn in vars_nota:
            par = df[[col_s, vn]].dropna()
            if len(par) < 10:
                row[vn] = np.nan
                continue
            rho, pv = spearmanr(par[col_s], par[vn])
            row[vn] = round(rho, 4)
            row[f"{vn}_p"] = round(pv, 6)
        rows.append(row)
    return pd.DataFrame(rows).set_index("Variável")


def correlacao_por_area(df, var_soc="QE_I08_NUM", var_nota="NT_GER_MEDIA", min_cursos=10):
    if var_soc not in df.columns or var_nota not in df.columns or "CO_GRUPO" not in df.columns:
        return pd.DataFrame()
    rows = []
    for grupo, sub in df.groupby("CO_GRUPO"):
        sub = sub[[var_soc, var_nota]].dropna()
        if len(sub) < min_cursos:
            continue
        rho, pv = spearmanr(sub[var_soc], sub[var_nota])
        rows.append({
            "CO_GRUPO": grupo, 
            "NOME_CURSO": CO_GRUPO_NOMES.get(grupo, f"Grupo {grupo}"),
            "N_CURSOS": len(sub), 
            "SPEARMAN_RHO": round(rho, 4), 
            "P_VALOR": round(pv, 6),
            "SIGNIFICATIVO": "Sim" if pv < 0.05 else "Não"
        })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values("SPEARMAN_RHO", ascending=False)


def heatmap_correlacoes_curso(df, salvar=True):
    interesse = [
        "QE_I08_NUM", "QE_I04_NUM", "QE_I05_NUM", "QE_I10_NUM", "QE_I17_NUM",
        "QE_I22_NUM_MEDIA", "QE_I23_NUM_MEDIA", "IDX_CAPITAL_CULT", "NU_IDADE",
        "PCT_SEXO_FEMININO", "PCT_INTERCAMBIO_SIM",
        "NT_GER_MEDIA", "NT_FG_MEDIA", "NT_CE_MEDIA"
    ]
    labs = {
        "QE_I08_NUM": "Renda", 
        "QE_I04_NUM": "Escol.Pai", 
        "QE_I05_NUM": "Escol.Mãe",
        "QE_I10_NUM": "Trabalho", 
        "QE_I17_NUM": "Escola.EM",
        "QE_I22_NUM_MEDIA": "Leitura", 
        "QE_I23_NUM_MEDIA": "H.Estudo",
        "IDX_CAPITAL_CULT": "Cap.Cultural", 
        "NU_IDADE": "Idade",
        "PCT_SEXO_FEMININO": "% Feminino", 
        "PCT_INTERCAMBIO_SIM": "% Intercâmbio",
        "NT_GER_MEDIA": "Nota Geral", 
        "NT_FG_MEDIA": "Form.Geral", 
        "NT_CE_MEDIA": "Comp.Espec."
    }
    cols = [v for v in interesse if v in df.columns]
    if len(cols) < 2:
        return
    
    sub = df[cols].dropna(thresh=max(1, len(cols) // 2))
    corr = sub.corr(method="spearman").rename(index=labs, columns=labs)
    
    fig, ax = plt.subplots(figsize=(12, 9))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    
    sns.heatmap(
        corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
        center=0, vmin=-1, vmax=1, linewidths=.5, ax=ax,
        cbar_kws={"label": "Spearman ρ"}
    )
    
    ax.set_title(
        "Correlação Spearman entre Cursos\n(Perfil Socioeconômico e Demográfico Médio × Desempenho Médio)",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    
    if salvar:
        caminho_png = str(OUTPUTS_DIR / "heatmap_correlacoes_curso.png")
        plt.savefig(caminho_png, dpi=150, bbox_inches="tight")
        log.info(f"[Heatmap] Gráfico salvo em: {caminho_png}")
        
    plt.show()
    plt.close()


def analisar_correlacoes(df):
    log.info("[Correlação] Calculando (nível curso)...")
    geral = correlacao_geral_curso(df)
    geral.to_csv(OUTPUTS_DIR / "correlacao_geral.csv", sep=";", decimal=",")
    
    por_area = correlacao_por_area(df)
    
    heatmap_correlacoes_curso(df, salvar=True)
    
    return geral, por_area


print("Funções de correlação (nível curso) com Heatmap definidas.")


Funções de correlação (nível curso) com Heatmap definidas.


In [11]:
df_corr_geral, df_corr_area = analisar_correlacoes(df_curso_eng)

[Correlação] Calculando (nível curso)...
[Heatmap] Gráfico salvo em: D:\TCC_ENADE\outputs\heatmap_correlacoes_curso.png


## Etapa 9 - Exportação em CSV para dashboard

In [12]:
COLUNAS_BASE_DASHBOARD = (
    ["NU_ANO", "CO_CURSO", "CO_IES", "CO_ORGACAD", "CO_GRUPO", "NOME_CURSO",
     "CO_UF_CURSO", "CO_REGIAO_CURSO", "N_ALUNOS",
     "PCT_PRESENCIAL", "PCT_SEXO_FEMININO", "PCT_INTERCAMBIO_SIM",
     "NU_IDADE", "QE_I08_NUM", "FAIXA_RENDA_SIMPL", "QE_I04_NUM", "QE_I05_NUM", "QE_I10_NUM",
     "QE_I17_NUM", "IDX_CAPITAL_CULT",
     "QE_I22_NUM_MEDIA", "QE_I22_NUM_MODA", "QE_I23_NUM_MEDIA", "QE_I23_NUM_MODA"]
    + VARS_DESEMPENHO_MEDIA
    + VARS_DESEMPENHO_MODA
)


def colunas_dashboard(df):
    base = [c for c in COLUNAS_BASE_DASHBOARD if c in df.columns]
    qtd = sorted(c for c in df.columns if c.startswith("QTD_"))
    return base + qtd


def exportar(df, ano=None, salvar_csv=False):
    sfx = f"_{ano}" if ano else ""
    cols = colunas_dashboard(df)
    df_exp = df[cols].copy()

    for col in VARS_DESEMPENHO_MEDIA + VARS_DESEMPENHO_MODA:
        if col in df_exp.columns:
            df_exp[col] = df_exp[col].round(1)

    for col in ["QE_I08_NUM", "QE_I04_NUM", "QE_I05_NUM", "QE_I10_NUM",
                "QE_I17_NUM", "IDX_CAPITAL_CULT",
                "QE_I22_NUM_MEDIA", "QE_I23_NUM_MEDIA"]:
        if col in df_exp.columns:
            df_exp[col] = df_exp[col].round(2)

    for col in ["PCT_PRESENCIAL", "PCT_SEXO_FEMININO",
                "PCT_INTERCAMBIO_SIM"]:
        if col in df_exp.columns:
            df_exp[col] = df_exp[col].round(4)

    csv_path = None

    if salvar_csv:
        csv_path = OUTPUTS_DIR / f"dashboard_export{sfx}.csv"
        df_exp.to_csv(
            csv_path,
            index=False,
            sep=";",
            decimal=",",
            encoding="utf-8-sig"
        )
        log.info(
            f"[Exporter] Dashboard CSV: {csv_path} "
            f"({len(df_exp):,} cursos, {len(cols)} colunas)"
        )

    return csv_path, df_exp


print("Função de exportação definida.")


Função de exportação definida.


## Etapa 10 - Pipeline multi-ano completo
Processa **todos** os anos encontrados em `data/` e gera um CSV consolidado com todos os anos.

In [13]:
import gc, traceback

resultados = []
frames_consolidado = []

for ano in ANOS_DISPONIVEIS:
    print(f"{'=' * 60}")
    print(f"  Processando {ano}...")
    print(f"{'=' * 60}")
    try:
        df_bruto_ano = ingerir_e_agregar(ano)
        df_eng_ano = adicionar_derivadas_curso(df_bruto_ano)
        del df_bruto_ano; gc.collect()

        _, df_exp_ano = exportar(df_eng_ano, ano=ano)
        del df_eng_ano; gc.collect()

        print(f"  {ano} processado: {len(df_exp_ano):,} cursos")

        frames_consolidado.append(df_exp_ano.copy())
        resultados.append({"ano": ano, "n_cursos": len(df_exp_ano)})

    except Exception as e:
        print(f"  Falha no ano {ano}: {e}")
        traceback.print_exc()
        continue

if frames_consolidado:
    df_consolidado = pd.concat(frames_consolidado, ignore_index=True)
    consolidado_path = OUTPUTS_DIR / "dashboard_consolidado_multianos.csv"
    df_consolidado.to_csv(consolidado_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
    print(f"\nConsolidado multi-ano: {len(df_consolidado):,} linhas (curso × ano) → {consolidado_path.name}")
else:
    print("\nNenhum ano processado com sucesso.")

pd.DataFrame(resultados)


[Ingestor] ENADE 2010 — 27 arquivos disponíveis


  Processando 2010...


[Ingestor] Arquivos não encontrados (ignorados): arq28, arq29
[Ingestor] 13 arquivos agregados → 4,221 cursos × 81 colunas
[Ingestor] ENADE 2011 — 27 arquivos disponíveis


  2010 processado: 4,221 cursos
  Processando 2011...


[Ingestor] Arquivos não encontrados (ignorados): arq28, arq29
[Ingestor] 13 arquivos agregados → 8,814 cursos × 81 colunas
[Ingestor] ENADE 2012 — 27 arquivos disponíveis


...

[Ingestor] 15 arquivos agregados → 9,896 cursos × 70 colunas
[Ingestor] ENADE 2023 — 32 arquivos disponíveis


  2022 processado: 9,896 cursos
  Processando 2023...


[Ingestor] 15 arquivos agregados → 9,812 cursos × 70 colunas


  2023 processado: 9,812 cursos

Consolidado multi-ano: 101,774 linhas (curso × ano) → dashboard_consolidado_multianos.csv


,ano,n_cursos
0,2010,4221
1,2011,8814
2,2012,7228
3,2013,3678
4,2014,9963
5,2015,8121
6,2016,4300
7,2017,10571
8,2018,8813
9,2019,8360
